In [ ]:
import sys 
sys.path.append('../')

In [ ]:
pwd

In [ ]:
cd ..

In [ ]:
import torch
import json
import os
from importlib.machinery import SourceFileLoader
from matplotlib import pyplot as plt

from pado.light import * 
from pado.optical_element import *
from pado.propagator import *

from utils.utils import *
from utils.Dirt import *

In [ ]:
import skimage

def visualize(tensor, cmap='hsv', vmin=None, vmax=None, figsize=None, axis=True, title=None, psnr=None, ssim=None, l1=None):
    plt.figure(figsize=figsize)
    if (vmin is not None) and (vmax is not None):
        plt.imshow(tensor[0].permute(1,2,0).cpu().detach().numpy(), cmap=cmap, vmin=vmin, vmax=vmax)
    else:
        plt.imshow(tensor[0].permute(1,2,0).cpu().detach().numpy(), cmap=cmap)
    if axis:
        plt.colorbar()
    else:
        plt.axis('off')
    title_string = title if title is not None else '' 
    title_string += f' l1: {l1:.3f}' if l1 is not None else ''
    title_string += f' psnr: {psnr:.3f}' if psnr is not None else ''
    title_string += f' ssim: {ssim:.3f}' if ssim is not None else ''
    plt.title(title_string)
    plt.show()

def grayscale(np_img):
    return (np_img[:,:,0]+np_img[:,:,1]+np_img[:,:,2])/3

def compute_psnr(GT, image):
    psnr = skimage.metrics.peak_signal_noise_ratio(
        GT[0].permute(1, 2, 0).cpu().detach().numpy(), image[0].permute(1, 2, 0).cpu().detach().numpy())
    return psnr

def compute_ssim(GT, image):
    ssim = skimage.metrics.structural_similarity(
        GT[0].mean(axis=0).cpu().detach().numpy(), image[0].mean(axis=0).cpu().detach().numpy())
    return ssim
    

In [ ]:
from scipy.io import savemat, loadmat

ckpt_path = './asset/ckpt/broadband'
args = json.load(open(ckpt_path + '/args.json'))
args = AttributeDict(args)

concat=False

args.device = 'cuda:1'
param = SourceFileLoader("param", os.path.join(ckpt_path,args.param_file.split('/')[-1])).load_module()
param = convert_resolution(param,args)
args.param = param
DOE_dir = ckpt_path+'/DOE_phase_02400.pt'
DOE_phase = torch.load(DOE_dir, map_location=args.device).detach()
DOE_phase = DOE_phase*edge_mask(param.R/2, cutoff=param.R/2, device=args.device).to(torch.float32).detach()
DOE_phase = ((DOE_phase + np.pi)%(2*np.pi) - np.pi).to(torch.float32)
DOE_train = DOE((1,1,param.R, param.R), param.DOE_pitch, param.material, wvl=param.DOE_wvl, device=args.device)
DOE_train.set_phase_change(DOE_phase.detach())  
DOE_train.visualize()

# visualize obstruction

In [ ]:
def compute_dirt(image_far, depth, args, predefined=False, perlin_noise_res=8):
    param = args.param
    
    if not hasattr(args, 'broadband') or not args.broadband:
        n_channels = len(param.wvls)
    elif args.broadband:
        n_channels = len(param.broadband_wvls)

    if predefined:
        import os
        import torch.nn.functional as F
        if n_channels==3:
            mask = torch.tensor(np.load(os.path.join(args.eval_path, 'mask_'+str(864)+'_0022.npy')), device=args.device).detach().to(torch.float32)
            image_near = torch.tensor(np.load(os.path.join(args.eval_path, 'img_near_'+str(864)+'_0022.npy')), device=args.device).detach().to(torch.float32)
            mask = F.interpolate(mask, (param.img_res, param.img_res))
            image_near = F.interpolate(image_near, (param.img_res, param.img_res))
            image_near = image_near * mask + image_far.to(args.device) * (1-mask)
            return image_near.to(args.device), mask.to(args.device)
        else:
            mask = torch.tensor(np.load(os.path.join(args.eval_path, 'mask_'+str(864)+'_0022.npy')), device=args.device).detach().to(torch.float32)
            mask = F.interpolate(mask, (param.img_res, param.img_res))
            if mask.shape[-3] > 1: # make mask 1-channel
                mask = mask[:,0:1,...]
            color_spectral = np.linspace(0, 1, n_channels).astype(np.float32)
            color_spectral = color_spectral * np.ones([param.img_res, param.img_res, n_channels])
            color_spectral = color_spectral.transpose(2, 0, 1)[np.newaxis, ...]
            image_near = torch.tensor(torch.from_numpy(color_spectral).to(args.device) * mask)
            image_near = image_near.to(args.device) * mask.to(args.device) + image_far * (1-mask.to(args.device))
            return image_near.to(args.device), mask.to(args.device)
    
    color_spectral = np.random.rand(n_channels)
    color_spectral = (color_spectral * np.random.uniform(0, 0.1) + 
                np.random.uniform(-0.02, 0.02, n_channels) )* np.ones([image_far.shape[-2], image_far.shape[-1], n_channels])
    perlin_noise = generate_fractal_noise_2d([image_far.shape[-2], image_far.shape[-1]], [perlin_noise_res,perlin_noise_res], tileable=(True,True), interpolant=interpolant)
    depth_adj = depth / param.depth_near_max
    T = transforms.Compose([transforms.ToTensor(),
                            transforms.RandomCrop(int(param.img_res * depth_adj)), 
                            transforms.Resize([image_far.shape[-2], image_far.shape[-1]])
                            ])
    perlin_noise = T(perlin_noise).squeeze().numpy()  
    alpha_map = perlin_noise * (perlin_noise > param.perlin_cutoff) * circle_grad(image_far.shape[-2], image_far.shape[-1])
    alpha_map /= np.max(alpha_map)
    image_near = torch.tensor(color_spectral* alpha_map[...,None]).permute(2,0,1)[None,...]
    mask = torch.tile(torch.tensor(1.0*(alpha_map[...,None] > 0.3)).permute(2,0,1)[None,...],(1,1,1,1)) 
    image_near = image_near.to(args.device) * mask.to(args.device) + image_far * (1-mask.to(args.device))
    return image_near.to(args.device), mask.to(args.device)

In [ ]:
from tqdm import tqdm

RGB=True
args.broadband = False 

param.wvls = [620e-9, 530e-9, 450e-9]
param.val_dir = './dataset/LIU4K_v2_validation_arbitrary'
image_far = F.interpolate(torch.tensor(plt.imread(os.path.join(param.val_dir, 'AJiyoon', 'IMG_3353.JPG')).transpose(2, 0, 1)[np.newaxis, ...]/255.0, device=args.device).to(torch.float32).detach(), (param.img_res, param.img_res))
image_near, mask = compute_dirt(image_far.get_intensity() if isinstance(image_far, pado.light.Light) 
                                else image_far, (param.depth_near_min + param.depth_near_max)/2, args, predefined=False, perlin_noise_res=8)
image_near = image_near.to(torch.float32)
mask = mask.to(torch.float32)

visualize(image_near, title='Near image with dirt mask', axis=False)

# Compute broadband PSF (accumulated PSF)

In [ ]:
from tqdm import trange 
import tqdm

def precompute_psfs(args, near_depths, far_depths, DOE_phase, normalizer, color_normalizer):
    """
    Compute PSF of each color channel, considering camera response, metasurface transmission, and bandpass filter.
    """
    param = args.param
    near_depth_psfs = list()
    far_depth_psfs = list()

    def _compute_psf(wvls, light, args, propagator, offset=(0, 0), normalize=True):
        param = args.param
        prop = Propagator(propagator)
        dim = light.field.shape
        aperture = Aperture(dim, param.DOE_pitch, param.aperture_diamter, param.aperture_shape, wvls, args.device)
        light = aperture.forward(light.clone())

        light_prop = prop.forward(light, param.sensor_dist, offset=offset, linear=False)
        psf = light_prop.get_intensity()

        # resize 
        psf = F.interpolate(psf, scale_factor=light_prop.pitch/(param.camera_pitch* param.image_sample_ratio), 
                            mode=args.resizing_method)
        if normalize:
            psf = psf / (torch.sum(psf, dim=(-2,-1), keepdim=True) + 1e-8) # normalize the psf for each wavelength
        return psf

    # compute near depth PSFs
    for near_depth_idx in range(len(near_depths)):
        near_depth = near_depths[near_depth_idx]
        psf_near_colors = [0, 0, 0]
        for wvl in tqdm.tqdm(param.full_broadband_wvls):
            light = pado.Light(DOE_phase.shape, pitch=param.DOE_pitch, wvl=wvl, device=args.device)
            light.set_spherical_light(near_depth)
            light.set_field(light.get_field()*torch.exp(1j*DOE_phase))
            psf_near = _compute_psf(wvl, light, args, 'SBL_ASM', offset=(0, 0), normalize=False)
            for color_idx, color_character in enumerate(['R', 'G', 'B']):
                psf_near_partial = psf_near/normalizer * (param.full_cam_response_func[wvl][color_character] * 
                                                          param.full_DOE_eff[wvl] * 
                                                          param.full_bandpass_filter_transmission[wvl])
                psf_near_colors[color_idx] = psf_near_colors[color_idx] + (psf_near_partial / color_normalizer[color_character])
        near_depth_psfs.append(torch.cat(psf_near_colors, dim=-3))

    # compute far depth PSFs
    for far_depth_idx in range(len(far_depths)):
        far_depth = far_depths[far_depth_idx]
        psf_far_colors = [0, 0, 0]
        for wvl in tqdm.tqdm(param.full_broadband_wvls):
            light = pado.Light(DOE_phase.shape, pitch=param.DOE_pitch, wvl=wvl, device=args.device)
            light.set_spherical_light(far_depth)
            light.set_field(light.get_field()*torch.exp(1j*DOE_phase))
            psf_far = _compute_psf(wvl, light, args, 'SBL_ASM', offset=(0, 0), normalize=False)
            for color_idx, color_character in enumerate(['R', 'G', 'B']):
                psf_far_partial = psf_far/normalizer * (param.full_cam_response_func[wvl][color_character] * 
                                                          param.full_DOE_eff[wvl] * 
                                                          param.full_bandpass_filter_transmission[wvl])
                psf_far_colors[color_idx] = psf_far_colors[color_idx] + (psf_far_partial / color_normalizer[color_character])
        far_depth_psfs.append(torch.cat(psf_far_colors, dim=-3))

    return near_depth_psfs, far_depth_psfs

color_normalizer = {'R':0, 'G':0, 'B':0}
for wvl in param.full_broadband_wvls:
    color_normalizer['R'] += param.full_cam_response_func[wvl]['R'] * param.full_DOE_eff[wvl] * param.full_bandpass_filter_transmission[wvl]
    color_normalizer['G'] += param.full_cam_response_func[wvl]['G'] * param.full_DOE_eff[wvl] * param.full_bandpass_filter_transmission[wvl]
    color_normalizer['B'] += param.full_cam_response_func[wvl]['B'] * param.full_DOE_eff[wvl] * param.full_bandpass_filter_transmission[wvl]

# Precompute PSF 
incident_light_intensity_sum = np.pi*((param.R/2)**2)*((param.DOE_pitch/param.camera_pitch)**2)
near_depth_psfs, far_depth_psfs = precompute_psfs(args, [0.05], [3], DOE_phase, incident_light_intensity_sum, color_normalizer)

In [ ]:
print('near depth psfs center')
for near_depth_psf in near_depth_psfs:
    visualize(near_depth_psf[:,0:1,...], cmap='magma')
    visualize(near_depth_psf[:,1:2,...], cmap='magma')
    visualize(near_depth_psf[:,2:3,...], cmap='magma')
print('far depth psfs center')
for far_depth_psf in far_depth_psfs:
    visualize(far_depth_psf[..., 0:1, far_depth_psf.shape[-2]//2-30 : far_depth_psf.shape[-2]//2+30, far_depth_psf.shape[-1]//2-30 : far_depth_psf.shape[-1]//2+30], cmap='magma')
    visualize(far_depth_psf[..., 1:2, far_depth_psf.shape[-2]//2-30 : far_depth_psf.shape[-2]//2+30, far_depth_psf.shape[-1]//2-30 : far_depth_psf.shape[-1]//2+30], cmap='magma')
    visualize(far_depth_psf[..., 2:3, far_depth_psf.shape[-2]//2-30 : far_depth_psf.shape[-2]//2+30, far_depth_psf.shape[-1]//2-30 : far_depth_psf.shape[-1]//2+30], cmap='magma')
print('far depth psfs')
for far_depth_psf in far_depth_psfs:
    visualize(far_depth_psf[:,0:1,near_depth_psf.shape[-2]//2-param.img_res//2 : near_depth_psf.shape[-2]//2+param.img_res//2, near_depth_psf.shape[-1]//2-param.img_res//2 : near_depth_psf.shape[-1]//2+param.img_res//2], cmap='magma')
    visualize(far_depth_psf[:,1:2,near_depth_psf.shape[-2]//2-param.img_res//2 : near_depth_psf.shape[-2]//2+param.img_res//2, near_depth_psf.shape[-1]//2-param.img_res//2 : near_depth_psf.shape[-1]//2+param.img_res//2], cmap='magma')
    visualize(far_depth_psf[:,2:3,near_depth_psf.shape[-2]//2-param.img_res//2 : near_depth_psf.shape[-2]//2+param.img_res//2, near_depth_psf.shape[-1]//2-param.img_res//2 : near_depth_psf.shape[-1]//2+param.img_res//2], cmap='magma')

# Normalize each color channel of PSF

In [ ]:
far_depth_psf = far_depth_psfs[0]
near_depth_psf = near_depth_psfs[0]
for c_idx in range(3):
    far_depth_psf[:,c_idx:c_idx+1,...] = far_depth_psf[:,c_idx:c_idx+1,...] / torch.sum(far_depth_psf[:,c_idx:c_idx+1,...])
    near_depth_psf[:,c_idx:c_idx+1,...] = near_depth_psf[:,c_idx:c_idx+1,...] / torch.sum(near_depth_psf[:,c_idx:c_idx+1,...])

# Broadband simulation visualization

In [ ]:
from tqdm import tqdm

RGB=True
args.broadband = False 

param.wvls = [620e-9, 530e-9, 450e-9]

convolved_far = fft_convolve2d_image_reflection(image_far, far_depth_psf)
visualize(convolved_far, vmin=0, vmax=1, figsize=(5, 5), axis=False, cmap='gray', 
        l1=abs(image_far-convolved_far).mean().item(),
        psnr=compute_psnr(image_far, convolved_far),
        ssim=compute_ssim(image_far, convolved_far))

visualize(convolved_far[:,0:1,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,0:1,...]-convolved_far[:,0:1,...]).mean().item(),
        psnr=compute_psnr(image_far[:,0:1,...], convolved_far[:,0:1,...]),
        ssim=compute_ssim(image_far[:,0:1,...], convolved_far[:,0:1,...]))
visualize(convolved_far[:,1:2,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,1:2,...]-convolved_far[:,1:2,...]).mean().item(),
        psnr=compute_psnr(image_far[:,1:2,...], convolved_far[:,1:2,...]),
        ssim=compute_ssim(image_far[:,1:2,...], convolved_far[:,1:2,...]))
visualize(convolved_far[:,2:3,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,2:3,...]-convolved_far[:,2:3,...]).mean().item(),
        psnr=compute_psnr(image_far[:,2:3,...], convolved_far[:,2:3,...]),
        ssim=compute_ssim(image_far[:,2:3,...], convolved_far[:,2:3,...]))

psf_center_sum = far_depth_psf[:, :, far_depth_psf.shape[-2]//2-30:far_depth_psf.shape[-2]//2+30, 
                               far_depth_psf.shape[-1]//2-30:far_depth_psf.shape[-1]//2+30].sum()/3
convolved_mask = torch.clamp(0.8*0.333/psf_center_sum*fft_convolve2d(mask, near_depth_psf), 0, 1) 
convolved_near = fft_convolve2d(image_near, near_depth_psf)

img_conv = convolved_far * (1 - convolved_mask) + convolved_near * convolved_mask
visualize(img_conv, vmin=0, vmax=1, figsize=(5, 5), axis=False, cmap='gray', 
        l1=abs(image_far-img_conv).mean().item(),
        psnr=compute_psnr(image_far, img_conv),
        ssim=compute_ssim(image_far, img_conv))

visualize(image_far, title='GT',axis=False)
visualize(image_near, title='GT obstruction',axis=False)


visualize(image_far[:,0:1,...], vmin=0, vmax=1, cmap='gray', axis=False)
visualize(image_far[:,1:2,...], vmin=0, vmax=1, cmap='gray', axis=False)
visualize(image_far[:,2:3,...], vmin=0, vmax=1, cmap='gray', axis=False)
visualize(img_conv[:,0:1,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,0:1,...]-img_conv[:,0:1,...]).mean().item(),
        psnr=compute_psnr(image_far[:,0:1,...], img_conv[:,0:1,...]),
        ssim=compute_ssim(image_far[:,0:1,...], img_conv[:,0:1,...]))
visualize(img_conv[:,1:2,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,1:2,...]-img_conv[:,1:2,...]).mean().item(),
        psnr=compute_psnr(image_far[:,1:2,...], img_conv[:,1:2,...]),
        ssim=compute_ssim(image_far[:,1:2,...], img_conv[:,1:2,...]))
visualize(img_conv[:,2:3,...], vmin=0, vmax=1, cmap='gray', axis=False,
          l1=abs(image_far[:,2:3,...]-img_conv[:,2:3,...]).mean().item(),
        psnr=compute_psnr(image_far[:,2:3,...], img_conv[:,2:3,...]),
        ssim=compute_ssim(image_far[:,2:3,...], img_conv[:,2:3,...]))

# PSF plane functions

In [ ]:
from tqdm import tqdm 
from pado.material import Material

def visualize_broadband_psf(psf_far, psf_near, wvl):
    psf_far = psf_far[0].permute(1,2,0).cpu().detach().numpy()
    psf_near = psf_near[0].permute(1,2,0).cpu().detach().numpy()
    psf_far_1d = psf_far.sum(axis=0)
    psf_near_1d = psf_near.sum(axis=0)
     
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax = axes[0, 0]
    im = ax.imshow(psf_far, vmax=0.01)  
    fig.colorbar(im, ax=ax)
    ax = axes[1, 0]
    im = ax.imshow(psf_near, vmax=0.0001)
    fig.colorbar(im, ax=ax)
    ax = axes[0, 1]
    im = ax.plot(psf_far_1d)  
    ax.set_ylim([0, 0.2])
    ax = axes[1, 1]
    im = ax.plot(psf_near_1d)
    ax.set_ylim([0, 0.1])
    plt.suptitle(f'wvl: {wvl*1e+9}nm')
    plt.tight_layout()
    plt.show()

def psf_XZ_vis(args, DOE_phase, depths, incident_z, wvl, normalizer):
    bl_asm = Propagator('SBL_ASM')
    DOE_train = DOE(DOE_phase.shape, args.param.DOE_pitch, material=Material('PDMS'), wvl=wvl, device=args.device)
    aperture = Aperture(DOE_phase.shape, args.param.DOE_pitch, args.param.DOE_pitch*DOE_phase.shape[-1], 'circle', wvl, args.device)
    DOE_train.set_phase_change(DOE_phase)  

    if args.use_lens:
        lens = RefractiveLens(DOE_phase.shape, args.param.DOE_pitch, args.param.focal_length, wvl=wvl, device=args.device)

    for idx, depth in tqdm(enumerate(depths)):
        light = Light(DOE_phase.shape, args.param.DOE_pitch, wvl,device=args.device)
        light.set_spherical_light(z=incident_z)
        light = aperture.forward(light.clone())
        light = DOE_train.forward(light.clone())
        if args.use_lens:
            light = lens.forward(light.clone())
        psf = bl_asm.forward(light.clone(), depth, offset=(0,0), linear=False).get_intensity()
        psf = psf[0, :, psf.shape[-2]//2-100:psf.shape[-2]//2+100, psf.shape[-1]//2-100:psf.shape[-1]//2+100].permute(1,2,0).cpu().detach().numpy()
        psf = psf/normalizer
        psf_1d = psf.sum(axis=0)
        if idx==0:
            psf_depth = np.zeros((psf.shape[0], len(depths)))
        psf_depth[:, idx] = psf_1d.squeeze()

    print(args.param.DOE_pitch, psf.shape[-1])

    plt.figure(figsize=(10, 5))
    plt.imshow(psf_depth, extent=[0, depths[-1]*1e3, 0, args.param.DOE_pitch*psf.shape[0]*1e6], interpolation='nearest', cmap='hot', aspect='auto')
    plt.xlabel('z (mm)')
    plt.ylabel('x (1um)')
    plt.title(f'wvl: {wvl*1e+9}nm, depth: {incident_z*1e+3}mm, z: {depths[0]*1e3}mm ~ {depths[-1]*1e3}mm')
    plt.tight_layout()
    plt.colorbar()
    plt.show()

def psf_Xdepths_vis(args, DOE_phase, wvl, incident_zs, z, normalizer, psf_width):
    psf_Xdepths = np.zeros((psf_width, len(incident_zs)))
    bl_asm = Propagator('SBL_ASM')
    DOE_train = DOE(DOE_phase.shape, args.param.DOE_pitch, material=Material('PDMS'), wvl=wvl, device=args.device)
    aperture = Aperture(DOE_phase.shape, args.param.DOE_pitch, args.param.DOE_pitch*DOE_phase.shape[-1], 'circle', wvl, args.device)
    DOE_train.set_phase_change(DOE_phase)  

    if args.use_lens:
        lens = RefractiveLens(DOE_phase.shape, args.param.DOE_pitch, args.param.focal_length, wvl=wvl, device=args.device)

    for idx, incident_z in tqdm(enumerate(incident_zs)):
        light = Light(DOE_phase.shape, args.param.DOE_pitch, wvl, device=args.device)
        light.set_spherical_light(z=incident_z)
        DOE_train.wvl = wvl
        aperture.wvl = wvl
        light = aperture.forward(light.clone())
        light = DOE_train.forward(light.clone())
        if args.use_lens:
            light = lens.forward(light.clone())
        psf = bl_asm.forward(light.clone(), z, offset=(0,0)).get_intensity()
        psf = F.interpolate(psf, scale_factor=args.param.DOE_pitch/args.param.camera_pitch, mode='area')
        psf = psf[0].permute(1,2,0).cpu().detach().numpy()
        psf = psf/normalizer
        psf_1d = psf.sum(axis=0)
        psf_Xdepths[:, idx] = psf_1d.squeeze()
    plt.figure()
    plt.imshow(psf_Xdepths, extent=[incident_zs[0]*1e2, incident_zs[-1]*1e2, 0, args.param.camera_pitch*psf.shape[-1]*1e8], interpolation='nearest')
    plt.xlabel('incident depth (cm)')
    plt.ylabel('x (10um)')
    plt.title(f'z: {z*1e+3}mm, wvl: {wvl*1e9}nm, depth: {incident_zs[0]*1e+2}cm ~ {incident_zs[-1]*1e+2}cm')
    plt.tight_layout()
    plt.colorbar()
    plt.show()

# xlambda plane, near and far (Synthetic training setup)

In [ ]:
def psf_Xwvls_vis(args, DOE_phase, wvls, incident_z, z, normalizer, incident_angle=(0, 0), Fraunhofer=False, vmin=None, vmax=None):
    if Fraunhofer:
        prop = Propagator('Fraunhofer')
    else:
        prop = Propagator('SBL_ASM_sparse')
    DOE_train = DOE(DOE_phase.shape, args.param.DOE_pitch, material=Material('PDMS'), wvl=param.DOE_wvl, device=args.device)
    aperture = Aperture(DOE_phase.shape, args.param.DOE_pitch, args.param.DOE_pitch*DOE_phase.shape[-1], 'circle', wvls[0], args.device)
    DOE_train.set_phase_change(DOE_phase)  

    dx, dy = -incident_z * np.tan(np.radians(incident_angle[0])), -incident_z * np.tan(np.radians(incident_angle[1]))
    prop_offset_x, prop_offset_y = args.param.focal_length * np.tan(np.radians(incident_angle[0])), args.param.focal_length * np.tan(np.radians(incident_angle[1]))

    for idx, wvl in tqdm(enumerate(wvls)):
        light = Light(DOE_phase.shape, args.param.DOE_pitch, wvl, device=args.device)
        light.set_spherical_light(z=incident_z, dx=dx, dy=dy)
        DOE_train.set_phase_change(DOE_phase) 
        DOE_train.wvl = wvl
        aperture.wvl = wvl
        light = aperture.forward(light.clone())
        light = DOE_train.forward(light.clone())
        psf_light = prop.forward(light.clone(), z, offset=(prop_offset_y, prop_offset_x), scale=(1, 1), linear=False)
        psf = psf_light.get_intensity()
        psf = psf[..., psf.shape[-2]//2-crop_width:psf.shape[-2]//2+crop_width, psf.shape[-1]//2-crop_width:psf.shape[-1]//2+crop_width]
        psf = F.interpolate(psf, scale_factor=psf_light.pitch/(pitch), mode='area')
        if Fraunhofer: 
            bw_r = light.get_bandwidth()[0]
            pitch_after_propagation = (np.array(wvls)) * np.array(param.sensor_dist / bw_r)
            scale_factors=pitch_after_propagation/(pitch * param.image_sample_ratio)
            scale_factors_max = max(scale_factors)
            max_R = int(scale_factors_max * param.R) # round down
            wl, wr = compute_pad_size(psf.shape[-1], max_R)
            hl, hr = compute_pad_size(psf.shape[-2], max_R)
            psf = F.pad(psf, (wl, wr, hl, hr), "constant", 0)
            psf = psf/psf.sum()
        else: 
            psf = psf/normalizer
        psf = psf[0].permute(1,2,0).cpu().detach().numpy()
        psf_1d = psf.sum(axis=1)
        if idx==0:
            psf_Xwvls = np.zeros((psf.shape[0], len(wvls)))
        psf_Xwvls[:, idx] = psf_1d.squeeze()
    plt.figure()
    if (vmin is None) or (vmax is None):
        plt.imshow(psf_Xwvls, extent=[wvls[0]*1e8, wvls[-1]*1e8, 0, args.param.DOE_pitch*args.param.R*1e4], interpolation='nearest')
    else:
        plt.imshow(psf_Xwvls, extent=[wvls[0]*1e8, wvls[-1]*1e8, 0, args.param.DOE_pitch*args.param.R*1e4], interpolation='nearest', vmin=vmin, vmax=vmax)
    plt.xlabel('wvl (10nm)')
    plt.ylabel('x (10mm)')
    plt.title(f'z: {z*1e+3}mm \n depth: {incident_z*1e+3}mm, \n wvl: {wvls[0]*1e9}nm ~ {wvls[-1]*1e9}nm, \n incident angle: {incident_angle}')
    plt.tight_layout()
    plt.colorbar()
    plt.show()
    return psf_Xwvls

pitch = param.camera_pitch
crop_width = 575 # 1438 for 4x magnification, 575 for 10x magnification

normalizer = np.pi*((param.R/2)**2)*((param.DOE_pitch/(pitch))**2)
wvls = np.linspace(350e-9, 700e-9, 71)

psf_Xwvls_dict = dict()
for incident_z in [0.04, 0.045, 100]:
    psf_Xwvls = psf_Xwvls_vis(args, DOE_phase, wvls, incident_z, param.focal_length, normalizer, (0,0), Fraunhofer=False)
    psf_Xwvls_dict[incident_z] = psf_Xwvls

for indident_z, psf_Xwvls in psf_Xwvls_dict.items():
    plt.figure()
    plt.imshow(psf_Xwvls, extent=[wvls[0]*1e8, wvls[-1]*1e8, 0, args.param.DOE_pitch*args.param.R*1e4], interpolation='nearest')
    plt.xlabel('wvl (10nm)')
    plt.ylabel('x (10mm)')
    plt.title(f'z: {param.focal_length*1e+3}mm \n depth: {indident_z*1e+3}mm, \n wvl: {wvls[0]*1e9}nm ~ {wvls[-1]*1e9}nm, \n incident angle: (0,0)')
    plt.tight_layout()
    plt.colorbar()
    plt.show()
